In [1]:
from data_utils import generate_training_prompts

In [2]:
import pandas as pd
from sklearn.model_selection import KFold

def get_cv_splits(data: pd.DataFrame, n_splits: int = 5, random_state: int = 42) -> list[tuple[pd.DataFrame, pd.DataFrame]]:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    cv_splits = []
    for train_idx, test_idx in kf.split(data):
        train_df = data.iloc[train_idx].copy()
        test_df = data.iloc[test_idx].copy()
        cv_splits.append((train_df, test_df))
    
    return cv_splits

path_to_data = "antonyms.json"
data = pd.read_json(path_to_data)
cv_pairs = get_cv_splits(data, n_splits=5, random_state=42)
data_train, data_test = cv_pairs[0]


In [3]:
data_train.head(10)

,input,target
0,flawed,perfect
1,orthodox,unorthodox
2,true,false
3,daily,nightly
4,distribution,concentration
5,valid,invalid
6,expand,contract
7,practical,impractical
8,privilege,disadvantage
9,mammoth,tiny


In [4]:
data_test.head()

,input,target
18,proceed,halt
25,privacy,publicity
29,professional,amateur
43,fascism,democracy
44,super,inferior


In [5]:
n_pairs_per_training_prompt = 10
training_prompts = generate_training_prompts(data_train, n_pairs_per_training_prompt, separator=', ')
print(training_prompts[0])

flawed:perfect, orthodox:unorthodox, true:false, daily:nightly, distribution:concentration, valid:invalid, expand:contract, practical:impractical, privilege:disadvantage, mammoth:


In [6]:
import torch
from transformer_lens import HookedTransformer

MODEL_NAME = "EleutherAI/gpt-j-6b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16"

model = HookedTransformer.from_pretrained_no_processing(
    model_name=MODEL_NAME, device=DEVICE, dtype=DTYPE
)
model.eval()


/root/miniconda3/envs/conceptors/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at EleutherAI/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.b

Loaded pretrained model EleutherAI/gpt-j-6b into HookedTransformer


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (ln1): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNorm(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
    

In [8]:
# def extract_activations_last_token(
#         model: HookedTransformer,
#         prompts: list[str],
#         extraction_layers: list[int]
# ) -> dict[int, torch.Tensor]:
#     """
#     Extract activations for the last token of each steering prompt from specific layers of the model.

#     Parameters:
#     model (HookedTransformer): The model used for generating text.
#     prompts (list): List of prompts to extract activations for.
#     extraction_layers (list): The layers from which activations are extracted.
#     device (str): The computing device (e.g., 'cuda', 'cpu').

#     Returns:
#     dict: A dictionary where each key is a layer number and each value is the
#         activations for the last token of each prompt. Shape: (n_prompts, n_activations).
#     """
#     activations_dict = {}
#     names = [f"blocks.{layer}.hook_resid_pre" for layer in extraction_layers]
#     cache, caching_hooks, _ = model.get_caching_hooks(lambda n: n in names)

#     with model.hooks(fwd_hooks=caching_hooks):
#         model.tokenizer.padding_side = "left"
#         _ = model(prompts)

#     for layer in extraction_layers:
#         prompt_activations = cache[f"blocks.{layer}.hook_resid_pre"].detach().cpu()
#         last_token_activations = prompt_activations[:, -1, :].squeeze()
#         activations_tensor = torch.tensor(
#             last_token_activations.numpy(), dtype=torch.float, device='cpu'
#         )
#         activations_dict[layer] = activations_tensor

#     return activations_dict

def extract_activations_last_token(
        model: HookedTransformer,
        prompts: list[str],
        extraction_layers: list[int],
        batch_size: int = 64,
) -> dict[int, torch.Tensor]:
    """
    Extract activations for the last token of each steering prompt from specific layers of the model.
    Processes prompts in batches to avoid memory issues.

    Parameters:
    model (HookedTransformer): The model used for generating text.
    prompts (list): List of prompts to extract activations for.
    extraction_layers (list): The layers from which activations are extracted.
    batch_size (int): Number of prompts to process at once.

    Returns:
    dict: A dictionary where each key is a layer number and each value is the
        activations for the last token of each prompt. Shape: (n_prompts, n_activations).
    """
    activations_dict = {layer: [] for layer in extraction_layers}
    names = [f"blocks.{layer}.hook_resid_pre" for layer in extraction_layers]
    
    # Process prompts in batches
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i+batch_size]
        cache, caching_hooks, _ = model.get_caching_hooks(lambda n: n in names)
        
        with model.hooks(fwd_hooks=caching_hooks):
            model.tokenizer.padding_side = "left"
            _ = model(batch_prompts)
            
        for layer in extraction_layers:
            prompt_activations = cache[f"blocks.{layer}.hook_resid_pre"].detach().cpu()
            last_token_activations = prompt_activations[:, -1, :].squeeze()
            # Handle the case where there's only one prompt in the batch
            if len(batch_prompts) == 1:
                last_token_activations = last_token_activations.unsqueeze(0)
            activations_dict[layer].append(last_token_activations)
        
        # Clear CUDA cache after each batch
        torch.cuda.empty_cache()
    
    # Concatenate the batched results
    for layer in extraction_layers:
        activations_dict[layer] = torch.cat(activations_dict[layer], dim=0)
    
    return activations_dict

EXTRACTION_LAYERS = list(range(9,17))
extraction_layers = EXTRACTION_LAYERS
activations_last_token = extract_activations_last_token(model, training_prompts, extraction_layers, batch_size=64)

OutOfMemoryError: CUDA out of memory. Tried to allocate 52.00 MiB. GPU 0 has a total capacity of 47.54 GiB of which 20.25 MiB is free. Process 2100134 has 47.49 GiB memory in use. Of the allocated memory 46.80 GiB is allocated by PyTorch, and 392.45 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
print(activations_last_token.keys())
print(activations_last_token[extraction_layers[0]].shape)
assert activations_last_token[extraction_layers[0]].shape == (len(training_prompts), model.cfg.d_model)

In [ ]:
def average_activations(
    activations: dict[int, torch.Tensor],
) -> dict[int, torch.Tensor]:
    """
    Precomputes averaged activations for all layers and experiments.

    Args:
    activations: A dictionary where each key is a layer number and each value is the
        activations for the last token of each prompt. Shape: (n_prompts, n_activations).

    Returns:
    averaged_activations: dictionary containing the averaged activations for each layer.
        Keys are layer_index and values are the averaged activations.
        Shape of values: (d_model,).
    """
    averaged_activations: dict[int, torch.Tensor] = {}

    for layer in activations.keys():
        # Extract the last-token activations of steering examples at the specified layer
        activation = activations[layer]
        # Compute the average activations
        avg_activation = torch.mean(activation, dim=0)
        # Store the average activations in the cache
        averaged_activations[layer] = avg_activation.detach().cpu()

    return averaged_activations

activations_last_token_averaged = average_activations(activations_last_token)
print(activations_last_token_averaged.keys())
print(activations_last_token_averaged[extraction_layers[0]].shape)
assert activations_last_token_averaged[extraction_layers[0]].shape == (model.cfg.d_model,)

In [ ]:
import pandas as pd
def generate_test_prompts(
    data: pd.DataFrame,
) -> list[str]:
    r"""
    Generates test prompts from the data.

    Args:
        data (pd.DataFrame): The data to generate training prompts from.
            Must have 'input' and 'output' columns.

    Returns:
        list[str]: A list of test prompts.
            Each training prompt is a string of the form "input:"
    """
    test_prompts: list[str] = []
    for i in range(len(data)):
        test_prompts.append(f"{data.iloc[i]['input']}:")
    
    return test_prompts

test_prompts = generate_test_prompts(data_test)
test_prompts[0:5]

In [27]:
from typing import Callable
def generate_hook_addition(steering_vector: torch.Tensor, beta: float) -> Callable:
    """
    Generates a hook function to add a steering vector to the last token.

    Parameters:
    - steering_vector (torch.Tensor): Steering vector.
    - beta (float): Scaling factor.

    Returns:
    - function: Hook function for adding steering vector.
    """

    def last_token_steering_hook(resid_pre, hook):
        for i in range(resid_pre.shape[0]):
            current_token_index = resid_pre.shape[1] - 1
            resid_pre[i, current_token_index, :] += steering_vector.squeeze().to(resid_pre.device) * beta

    return last_token_steering_hook

def generate_text(model: HookedTransformer, prompts: list[str], hooks: list[Callable], max_new_tokens: int = 5, temperature: float = 0) -> list[str]:
    with model.hooks(fwd_hooks=hooks):
        # for prompt in prompts:
        #     result = model.generate(prompt, max_new_tokens=max_new_tokens, temperature=temperature)
        #     results.append(result)
        #     torch.cuda.empty_cache()
        results = model.generate(prompts, max_new_tokens=max_new_tokens, temperature=temperature, padding_side='left')
    
    # remove the prompts from the results
    results = [result[len(prompt):] for result, prompt in zip(results, prompts)]

    return results

In [ ]:
layer = extraction_layers[0]
beta = 1
addition_hook = generate_hook_addition(steering_vector=activations_last_token_averaged[layer], beta=beta)
activation_modification = (f"blocks.{layer}.hook_resid_pre", addition_hook)
hooks = [activation_modification]

predictions = generate_text(model, test_prompts, hooks)


In [ ]:
for test, prediction in zip(test_prompts[110:115], predictions[110:115]):
    print(f"{test}{prediction}")
    print()

In [ ]:
# function to calculate the accuracy of the predictions

def calculate_accuracy(predictions: list[str], targets: pd.Series) -> float:
    correct = 0
    for prediction, target in zip(predictions, targets):
        if prediction.startswith(target):
            correct += 1
    return correct / len(predictions)

calculate_accuracy(predictions, data_test['target'])

## Loop through hyperparameters and splits